# Qwen3-4B JP→EN LoRA Fine-tune on H100 80GB
LoRA fine-tune (PEFT), loss only on assistant turns, No flash Attn, bf16.

## 1. Install dependencies

In [1]:
#%pip install -r requirements.txt

## 2. Imports

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Must be before any torch import
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model, TaskType

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA H100 80GB HBM3
VRAM: 85.0 GB


## 3. Load & inspect dataset

In [ ]:
#jsonl sample {"messages": [{"role": "system", "content": "You are a careful literary translation model. Translate faithfully, preserve tone, punctuation, names, and paragraph structure."}, {"role": "user", "content": "使徒の襲撃、及び王都侵攻\n\n\n\n薄暗く明かり一つ無い部屋の中に、格子の嵌った小さな窓から月明かりだけが差し込んで黒と白のコントラストを作り出していた。\n\n部屋の中は酷く簡素な作りになっている。\n\n鋼鉄造りの六畳一間、木製のベッドにイス、小さな机、そしてむき出しのトイレ。"}, {"role": "assistant", "content": "In a room where the only light source was produced by the moonlight rays, causing a contrast of black and white from the narrow grate window.\n\nA simple and plain room can be seen.\n\nIt’s only around 6 tatami mats in size with a small desk, chair, wooden bed, and a simple toilet."}], "novel": "Arifureta", "chapter": "100.txt", "direction": "jp2en", "window_size": 3, "window_pair_indices": [0, 1, 2], "task": "Translate the Japanese text into natural English. Preserve paragraph breaks and meaning.", "source_side": "jp", "target_side": "en"}

DATA_URL = "./training.jsonl"

ds = load_dataset("json", data_files={"train": DATA_URL}, split="train")
print(f"Total samples: {len(ds)}")
print(ds[0])

Total samples: 238871
{'messages': [{'role': 'system', 'content': 'You are a careful literary translation model. Translate faithfully, preserve tone, punctuation, names, and paragraph structure.'}, {'role': 'user', 'content': '使徒の襲撃、及び王都侵攻\n\n\n\n薄暗く明かり一つ無い部屋の中に、格子の嵌った小さな窓から月明かりだけが差し込んで黒と白のコントラストを作り出していた。\n\n部屋の中は酷く簡素な作りになっている。\n\n鋼鉄造りの六畳一間、木製のベッドにイス、小さな机、そしてむき出しのトイレ。'}, {'role': 'assistant', 'content': 'In a room where the only light source was produced by the moonlight rays, causing a contrast of black and white from the narrow grate window.\n\nA simple and plain room can be seen.\n\nIt’s only around 6 tatami mats in size with a small desk, chair, wooden bed, and a simple toilet.'}], 'novel': 'Arifureta', 'chapter': '100.txt', 'direction': 'jp2en', 'window_size': 3, 'window_pair_indices': [0, 1, 2], 'task': 'Translate the Japanese text into natural English. Preserve paragraph breaks and meaning.', 'source_side': 'jp', 'target_side': 'en'}


## 4. Filter to JP→EN only and deduplicate
Only keep `direction == 'jp2en'` rows, then drop exact duplicate windows.

In [4]:
ds = ds.filter(lambda x: x.get("direction") == "jp2en")
print(f"After direction filter: {len(ds)}")

# Deduplicate on novel + chapter + window_pair_indices
seen = set()
def is_unique(example):
    key = (example["novel"], example["chapter"], str(example["window_pair_indices"]))
    if key in seen:
        return False
    seen.add(key)
    return True

ds = ds.filter(is_unique)
print(f"After dedup: {len(ds)}")

After direction filter: 238871
After dedup: 238871


## 5. Train / eval split
5% held out for eval loss tracking.

In [5]:
ds = ds.train_test_split(test_size=0.05, seed=42)
train_ds = ds["train"]
eval_ds  = ds["test"]

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")

Train: 226927 | Eval: 11944


## 6. Load tokenizer & base model
Load base model in bf16 with Flash Attention 2. Weights stay frozen —
only the LoRA adapter matrices will be trained.

In [6]:
MODEL_ID = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # needed for packing

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    trust_remote_code=True,
)
model.config.use_cache = False  # required during training
print(f"Total model params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Total model params: 4.02B


In [7]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    # Qwen3 attention + MLP projection names
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~1-2% of total params trainable

trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157


## 11. Trainer
Using SFTTrainer with `assistant_only_loss=True` which provides built in automatic encoding support for qwen chat style.

In [9]:
import os
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./qwen3-4b-jp2en-lora",
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    max_length=900,
    dataset_text_field="text",
    assistant_only_loss=True, 
    packing=False,
    per_device_train_batch_size=48,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    bf16=True,
    loss_type="chunked_nll",
    tf32=True,
    optim="adamw_torch_fused",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=8,
    dataloader_pin_memory=True,
    eval_strategy="no",
    logging_steps=25,
    report_to="none",
)


trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

Tokenizing train dataset:   0%|          | 0/226927 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/11944 [00:00<?, ? examples/s]

## 12. Train

In [ ]:
import os
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = "./qwen3-4b-jp2en-lora"

# Find latest checkpoint
last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found. Starting fresh training.")
    trainer.train()

# Save final LoRA adapter
FINAL_DIR = "./qwen3-4b-jp2en-lora-final"

model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print(f"Training complete. LoRA adapter saved to {FINAL_DIR}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


No checkpoint found. Starting fresh training.


Step,Training Loss
25,4.900400
50,2.039200
75,1.760700


In [ ]:
import requests
from requests.structures import CaseInsensitiveDict

url = "https://trigger.macrodroid.com/2a3f03e4-c9a3-462c-8ac1-d713a586db99/notify-the-great-avinash"

headers = CaseInsensitiveDict()
headers["Content-Type"] = "application/json"

data = """
{
 "description": "I am god"
}
"""


resp = requests.post(url, headers=headers, data=data)

print(resp.status_code)

200


In [ ]:
from transformers import TrainerCallback

class ShapeLogger(TrainerCallback):
    def on_step_begin(self, args, state, control, **kwargs):
        inputs = kwargs.get("inputs")
        if inputs is not None:
            print(
                f"step={state.global_step} "
                f"shape={inputs['input_ids'].shape}"
            )

trainer.add_callback(ShapeLogger())

In [ ]:
def inspect(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False
    )
    return {"len": len(tokenizer(text)["input_ids"])}

lens = train_ds.map(inspect)   # replace with your dataset variable

print("max:", max(lens["len"]))
print("top20:", sorted(lens["len"])[-20:])

## 13. (Optional) Merge LoRA adapter into base model
Merging produces a single standalone model with no PEFT dependency at inference time.
Skip this if you want to keep the adapter separate (e.g. to swap adapters).

In [ ]:
from peft import PeftModel

# Reload base in bf16, then merge
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    trust_remote_code=True,
)
merged_model = PeftModel.from_pretrained(base_model, "./qwen3-4b-jp2en-lora-final")
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained("./qwen3-4b-jp2en-merged")
tokenizer.save_pretrained("./qwen3-4b-jp2en-merged")
print("Merged model saved to ./qwen3-4b-jp2en-merged")

## 14. Quick inference test
Sanity check — give it a JP sentence and see the EN output.
Uses the adapter-only checkpoint (no merge required).

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="./qwen3-4b-jp2en-lora-final",
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)

test_jp = "彼女は静かに立ち上がり、窓の外を眺めた。"

messages = [
    {"role": "system", "content": "You are a careful literary translation model. Translate faithfully, preserve tone, punctuation, names, and paragraph structure."},
    {"role": "user",   "content": test_jp},
]

out = pipe(messages, max_new_tokens=256, do_sample=False)
print(out[0]["generated_text"][-1]["content"])